In [ ]:
# ── Colab Setup (run this first) ──────────────────────────────────────────
!pip install openai faiss-cpu tiktoken -q

import os
# Option 1: Paste your key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 02 — Vector Store
AutoRFP-AI: Build FAISS Retrieval Index

WHAT THIS DOES:
- Embeds all 50 Q&A pairs using OpenAI embeddings
- Builds a FAISS index for fast similarity search
- Implements the retrieve() function with dedup support
- Tests retrieval + deduplication logic

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITE: Run Notebook 01 first

### Setup

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from openai import OpenAI
import faiss

client = OpenAI()
DATA_DIR = Path("data")

with open(DATA_DIR / "past_responses.json") as f:
    knowledge_base = json.load(f)
print(f"Loaded {len(knowledge_base)} Q&A pairs")

### Create embeddings

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"  # 1536 dims, fast, cheap


def get_embeddings(texts):
    """Batch embed texts using OpenAI API."""
    response = client.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    return [item.embedding for item in response.data]


# Combine question + answer for richer retrieval
texts_to_embed = [
    f"QUESTION: {qa['question']}\nANSWER: {qa['answer']}"
    for qa in knowledge_base
]

print(f"Embedding {len(texts_to_embed)} documents...")
embeddings = get_embeddings(texts_to_embed)
print(f"Got {len(embeddings)} embeddings, dim={len(embeddings[0])}")

### Build FAISS index

In [ ]:
embedding_dim = len(embeddings[0])
embedding_matrix = np.array(embeddings, dtype="float32")

# Normalize for cosine similarity (IndexFlatIP on normalized = cosine)
faiss.normalize_L2(embedding_matrix)

index = faiss.IndexFlatIP(embedding_dim)
index.add(embedding_matrix)

print(f"FAISS index: {index.ntotal} vectors x {embedding_dim} dims")

### Build retrieve() function

In [ ]:
def retrieve(query, top_k=3, exclude_hashes=None):
    """
    Find the top-k most similar past RFP answers.

    Args:
        query: New RFP question to match against
        top_k: Number of results
        exclude_hashes: Set of content hashes to skip (for dedup in retry loop)

    Returns:
        List of dicts: id, question, answer, category, score, hash
    """
    if exclude_hashes is None:
        exclude_hashes = set()

    query_vec = np.array(get_embeddings([query]), dtype="float32")
    faiss.normalize_L2(query_vec)

    search_k = min(top_k + len(exclude_hashes) + 5, index.ntotal)
    scores, indices = index.search(query_vec, search_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        qa = knowledge_base[idx]
        if qa["hash"] in exclude_hashes:
            continue
        results.append({
            "id": qa["id"], "question": qa["question"],
            "answer": qa["answer"], "category": qa["category"],
            "score": float(score), "hash": qa["hash"],
        })
        if len(results) >= top_k:
            break
    return results


print("Retriever ready")

### Test retrieval

In [ ]:
test_queries = [
    "What encryption standards do you use for data at rest and in transit?",
    "Describe your disaster recovery and business continuity plan.",
    "What is your pricing model for 500+ enterprise users?",
]

print("Testing retrieval...\n")
for query in test_queries:
    print(f"Q: {query}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] (score={r['score']:.3f}) [{r['category']}] {r['question'][:80]}...")
    print()

### Test deduplication (the bug fix!)

In [ ]:
print("Testing deduplication logic...\n")
query = "What security certifications do you hold?"

# Round 1 — no exclusions
round_1 = retrieve(query, top_k=3)
print("Round 1:")
seen = set()
for r in round_1:
    print(f"  [{r['hash']}] {r['question'][:70]}...")
    seen.add(r["hash"])

# Round 2 — exclude what we already saw
round_2 = retrieve(query, top_k=3, exclude_hashes=seen)
print(f"\nRound 2 (excluding {len(seen)} hashes):")
for r in round_2:
    print(f"  [{r['hash']}] {r['question'][:70]}...")

overlap = {r["hash"] for r in round_1} & {r["hash"] for r in round_2}
print(f"\nOverlap: {len(overlap)} (should be 0)")

### Save index

In [ ]:
faiss.write_index(index, str(DATA_DIR / "faiss_index.bin"))

with open(DATA_DIR / "retriever_meta.json", "w") as f:
    json.dump({
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dim": embedding_dim,
        "num_documents": len(knowledge_base),
    }, f, indent=2)

print(f"\nSaved FAISS index and metadata")
print("Ready for Notebook 03 (Agent Pipeline)")